In [1]:
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
import torch
import numpy as np
from PIL import Image
import requests
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
from datetime import datetime
import logging
import supervision as sv
import pathlib
from tqdm import tqdm
import pandas as pd
from collections import defaultdict

In [2]:
# Initialize the camera matrix and distortion coefficients, speed zone
fx = 480.35583204
fy = 437.11894211
cx = 354.90375117
cy = 249.48026187

dis_vec = np.array([-4.09277234e-01, 1.89529196e-01, -2.03433425e-04, -2.01818939e-03, -5.22683299e-02])

# Create camera matrix
camera_matrix = np.array([
    [fx, 0, cx],
    [0, fy, cy],
    [0, 0, 1]
])

speed_zone = [[109,65],[263,211],[60,248],[5,74]]

In [3]:
# function for undistorting the frame and crop it

def undistort(frame):
    h, w = frame.shape[:2]
    newcameramtx, roi = cv2.getOptimalNewCameraMatrix(camera_matrix, dis_vec, (w, h), 1, (w, h))
    dst = cv2.undistort(frame, camera_matrix, dis_vec, None, newcameramtx)
    x, y, w, h = roi
    dst = dst[y:y + h, x:x + w]

    dst = Image.fromarray(dst)

    return dst

In [4]:
# function to get the depth map from the frist frame

def get_depth_map(frame):

    CHECKPOINT = "depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf"


    image_processor = AutoImageProcessor.from_pretrained(CHECKPOINT)
    model = AutoModelForDepthEstimation.from_pretrained(CHECKPOINT)

    inputs = image_processor(images=frame, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)
        predicted_depth = outputs.predicted_depth

    prediction = torch.nn.functional.interpolate(
        predicted_depth.unsqueeze(1),
        size=frame.size[::-1],
        mode="bicubic",
        align_corners=False,
    ).squeeze(0).squeeze(0).cpu().numpy()

    return prediction

In [5]:
# calculate distance between two points using depth map and camera matrix

def calculate_distance(depth_map, camera_matrix, x1, y1, x2, y2):
    # Get the depth values at the two points
    z1 = depth_map[y1, x1]
    z2 = depth_map[y2, x2]
    
    # Calculate the 3D coordinates of the two points
    p1 = np.array([(x1 - camera_matrix[0, 2]) * z1 / camera_matrix[0, 0],
                   (y1 - camera_matrix[1, 2]) * z1 / camera_matrix[1, 1],
                   z1])
    p2 = np.array([(x2 - camera_matrix[0, 2]) * z2 / camera_matrix[0, 0],
                   (y2 - camera_matrix[1, 2]) * z2 / camera_matrix[1, 1],
                   z2])
    
    # Calculate the distance between the two points
    distance = np.linalg.norm(p2 - p1)/1.5
    
    return distance

In [6]:
# getting depth map from the first frame

image = Image.open('../test1.jpg')
frame = np.array(image)
frame = undistort(frame)
depth_map = get_depth_map(frame)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


# Speed calculation for cars in video

In [7]:
# initiate yolo11 for car detection and tracking
model = YOLO("yolo11l.pt")
confidence_threshold = 0.1
tracker_config = "../configs/bytetrack.yml"

In [8]:
def get_count(frame ,start_time, config, speeds, model, speed_zone, crossed_objects, depth_map,
              track_history,count) -> dict:

	results, boxes, class_ids, class_names, annotated_frame = get_result(frame, config, model)

	if results[0].boxes.id is not None:
		track_ids = results[0].boxes.id.cpu().int().tolist()

		# Plot the tracks and count objects crossing the line
		for box, track_id, cls in zip(boxes, track_ids, class_names):
			x, y, w, h = box
			pt = (int(x.numpy()), int(y.numpy()))
			cls = cls
			track = track_history[track_id]
			track.append((float(x), float(y)))  # x, y center point


			if len(track) > 30:  # retain 30 tracks for 30 frames
					track.pop(0)
			# Check if the object crosses the line
			if track_id not in crossed_objects["EB"]:
				time_seen = datetime.fromtimestamp(int(count/30) + start_time.timestamp())
				crossed_objects["EB"][track_id] = [time_seen.strftime("%Y-%m-%d %H:%M:%S"), cls]

			# calculate the speed if object in speed zone based on track history
			if len(track) > 15:
				distance = calculate_distance(depth_map, camera_matrix, int(track[0][0]), int(track[0][1]), int(track[-1][0]), int(track[-1][1]))
				time = len(track)/30
				speed = distance/time
				speeds[track_id] = speed * 3.6

				# show speed on the frame
				if track_id in speeds:
					cv2.putText(annotated_frame, f"{speeds[track_id]:.2f} km/h", (int(x - w / 2), int(y - h / 2)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
				
				# Annotate the object as it crosses the line
				cv2.rectangle(annotated_frame, (int(x - w / 2), int(y - h / 2)), (int(x + w / 2), int(y + h / 2)), (0, 255, 0), 2)

		# Annotate center of the object
		cv2.circle(annotated_frame, pt, 5, (0, 255, 0), -1)

	return annotated_frame

def get_result(frame, config, model) -> dict:

	class_ids = list(config["class_ids"].keys())
	results = model.track(frame, classes=class_ids, persist=True, save=False, tracker=config["tracker_config"],
								verbose=False, conf = config["conf"], iou = config["iou"], agnostic_nms = False)


	# Get the boxes and track IDs
	boxes = results[0].boxes.xywh.cpu()

	class_ids = results[0].boxes.cls.cpu().int().tolist()

	class_names = [config["class_ids"][i] for i in class_ids]

	# Visualize the results on the frame
	annotated_frame = results[0].plot()

	return (results, boxes, class_ids, class_names, annotated_frame)

def resample_data(df_first: pd.DataFrame, interval: str = '1min') -> pd.DataFrame:

	df = df_first.copy()
	# Reset the index to make the track_id a column
	df.reset_index(inplace=True)
	# Set the index to the timestamp
	df['timestamp'] = pd.to_datetime(df['timestamp'])
	df.set_index('timestamp', inplace=True)
	df.rename(columns={'index': 'Track ID'}, inplace=True)
	# Resample the data by intervals
	resampled_df = df.resample(interval).count()

	# Reset the index to make the time intervals a column
	resampled_df = resampled_df.reset_index()

	# Rename the time interval column to 'timestep'
	resampled_df.rename(columns={'index': 'timestep'}, inplace=True)

	return resampled_df.copy()
	
def generate_report(uuid,crossed_objects) -> pd.DataFrame:

	print(crossed_objects)

	df_1 = pd.DataFrame.from_dict(crossed_objects["EB"], orient='index', columns=['timestamp', 'Class'])

	df_1 = resample_data(df_1)


	df_1['Direction'] = 'EB'

	resampled_df = df_1.copy()
	resampled_df['uuid'] = uuid

	print("df", resampled_df)

	return resampled_df

In [9]:
start_time = datetime.strptime("202410011845","%Y%m%d%H%M")
file_name = "Standard_SCU2VN_2024-10-01_2345.020.mp4"   
file_path = "../Standard_SCU2VN_2024-10-01_2345.020.mp4"
track_history = defaultdict(lambda: [])
crossed_objects = {"EB": {}}
frame_count = 0
speeds = {}
class_ids = {2:'car', 5:'bus', 7:'truck'}

config = {
	"class_ids": class_ids,
	"tracker_config": tracker_config,
	"conf": confidence_threshold,
	"iou": 0.5
}

cap = cv2.VideoCapture(file_path)
assert cap.isOpened(), "Error reading video file"
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))


logging.info("frame_width: " + str(w))
logging.info("frame_height: " + str(h))

frame_generator = sv.get_video_frames_generator(source_path=file_path)

# Open a video sink for the output video
video_info = sv.VideoInfo.from_video_path(file_path)
if not pathlib.Path("../video/").exists():
    pathlib.Path("../video/").mkdir(parents=True, exist_ok=True)

video_report_path = "../video/" + "test_1.mp4"
with sv.VideoSink(video_report_path, video_info) as sink:
	for frame in tqdm(frame_generator, total=video_info.total_frames):
		success, frame = cap.read()
		frame_count += 1
		if success:
			# undistort the frame
			frame = undistort(frame)
			annotated_frame = get_count(frame, start_time, config, speeds, model, speed_zone, crossed_objects, depth_map, track_history, frame_count)

			# convert annotated frame to distored frame
			annotated_frame = np.array(annotated_frame)
			annotated_frame = cv2.resize(annotated_frame, (w, h))

			# Show the frame with annotations
			cv2.imshow("Frame", annotated_frame)
			cv2.waitKey(1)


			# Draw the line on the frame
			cv2.polylines(annotated_frame, [np.array(speed_zone, np.int32)], True, (0, 255, 0), 2)

			# Write the count of objects on each frame
			count_text_1 = f"Objects crossed EB: {len(crossed_objects['EB'])}"
			cv2.putText(annotated_frame, count_text_1, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

			# Write the frame with annotations to the output video
			sink.write_frame(annotated_frame)
			
		else:
			pass

# Release the video capture
cap.release()
print(f"Data has been written to {video_report_path}")


  8%|▊         | 828/10792 [03:45<45:18,  3.67it/s]


KeyboardInterrupt: 